# 53 — Phase 1 Bundle A × v5-kto MERGE on Blind-A

**Goal**: clean apples-to-apples test of BGE-M3 with the responder held constant (v5-kto). Config 122 was confounded — it changed both the retriever AND the responder. This config (132) changes ONLY the metadata-dense embedder vs the prior 2026-05-15 v5-kto submission.

**Setup**:
- Retriever: BGE-M3 metadata-dense + lyrics-Qwen3-0.6B + BM25 + ProRank + CMQR (same retriever stack as 122)
- Responder: **v5-kto-merged 3B** (same as your 2026-05-15 0.21-composite submission)
- Dataset: Blind-A (80 unique sessions × 1 turn = 80 turns)

**Single-axis change vs prior v5-kto submission**: only metadata-dense embedder (Qwen3-Embedding-0.6B → BGE-M3). Everything else identical.

**Decision rule**:
| Outcome | Action |
|---|---|
| Composite ≥ 0.23 (≥+0.02 over prior 0.21) | BGE-M3 helps — ship; move to Bundle B (Qwen3-4B) next week |
| Composite in 0.21 ± 0.05 (ambiguous) | BGE-M3 is roughly neutral. Keep in ensemble for future combinations; don't pursue solo |
| Composite < 0.16 (≥−0.05 below prior) | BGE-M3 hurts. Pivot to Bundle B or preprocessing |

**Wallclock on L4** (v5-kto is 3B vs 122's 1.5B responder, so longer):
| Step | Time | Cached? |
|---|---|---|
| BGE-M3 catalog embed | Skip — done in your previous session (Drive) | yes |
| Download v5-kto 3B from HF | ~3–5 min | yes (HF cache on Drive) |
| Inference (80 turns) | ~45–90 min | no |
| Package zip | ~10 sec | — |
| **Total** | **~1–2 hr** | |

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth + Drive (BGE-M3 catalog cache + v5-kto model weights persist on Drive).
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('NO HF_TOKEN — set it in Colab secrets before cell 5.', e)

from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
print('HF_HOME =', os.environ['HF_HOME'])

DRIVE_EMBED_DIR = '/content/drive/MyDrive/recsys2026_embed_cache'
os.makedirs(DRIVE_EMBED_DIR, exist_ok=True)
# Correct symlink path (matches configs' cache_dir resolution).
REPO_CACHE_DIR = '/content/recsys2026/experiments/cache/dense_local'
os.makedirs(os.path.dirname(REPO_CACHE_DIR), exist_ok=True)
if os.path.islink(REPO_CACHE_DIR) or os.path.exists(REPO_CACHE_DIR):
    !rm -rf {REPO_CACHE_DIR}
!ln -s {DRIVE_EMBED_DIR} {REPO_CACHE_DIR}
print('symlinked', REPO_CACHE_DIR, '->', DRIVE_EMBED_DIR)

In [ ]:
# 4) Install deps.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml bm25s scipy numpy sentence-transformers

In [ ]:
# 5) Verify BGE-M3 catalog cache exists from previous session. Re-embed if missing.
import os
EMBED_CACHE = f"{REPO_CACHE_DIR}/BAAI_bge-m3/bge-m3-metadata/track_embeddings.pkl"
if os.path.exists(EMBED_CACHE):
    print(f'BGE-M3 cache OK at {EMBED_CACHE}')
else:
    print('No cache — re-embedding catalog (~1-2 hr on L4).')
    !python scripts/embed_catalog.py \
        --model BAAI/bge-m3 \
        --label bge-m3-metadata \
        --batch-size 64

In [ ]:
# 6) Run merged config 132 (BGE-M3 retriever + v5-kto responder) on Blind-A.
# batch_size=16 because v5-kto is 3B (vs Qwen-1.5B in config 122 which used batch=32).
%cd /content/recsys2026/music-crs-baselines
TID = '132-bge-m3-v5kto-prorank-rerank-blindsetA'
PRED = f'exp/inference/blindset_A/{TID}.json'
import os
if os.path.exists(PRED):
    print(f'Predictions exist at {PRED}. To force re-run: !rm', PRED)
else:
    print(f'Running merged config 132 on Blind-A (~45-90 min on L4)...')
    !python run_inference_blindset.py --tid {TID} --batch_size 16
%cd /content/recsys2026

In [ ]:
# 7) Validate schema + package CodaBench-ready zip.
from datetime import date
import sys, os, shutil
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

PRED_ABS = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/132-bge-m3-v5kto-prorank-rerank-blindsetA.json'
ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_132_bge_m3_v5kto.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)

predictions = load_prediction(PRED_ABS)
errors = validate_schema(predictions, 'blindA')
if errors:
    print('SCHEMA FAILED:')
    for e in errors[:20]: print(f'  - {e}')
    raise SystemExit('Refusing to package invalid predictions.')

package_zip(PRED_ABS, ZIP_PATH)
print(f'\nReady for CodaBench: {ZIP_PATH}')
print(f'  size: {os.path.getsize(ZIP_PATH):,} bytes, {len(predictions)} predictions')

DRIVE_SUBS = '/content/drive/MyDrive/recsys2026_submissions'
os.makedirs(DRIVE_SUBS, exist_ok=True)
drive_copy = os.path.join(DRIVE_SUBS, os.path.basename(ZIP_PATH))
shutil.copy(ZIP_PATH, drive_copy)
print(f'  also at: {drive_copy}')

## Submit
1. Download zip from `/content/drive/MyDrive/recsys2026_submissions/`
2. Upload to CodaBench → wait ~5 min for leaderboard
3. Compare composite to your prior v5-kto 0.21

This is your 3rd Blind-A slot this week (cap=3) — the result decides next week's direction (Bundle B vs preprocessing vs other).